In [1]:
import os  # For navigating through folders and accessing files.
import cv2  # The computer vision library that does all the heavy lifting.
import numpy as np

In [2]:
# Load .tif as numpy.ndarray with shape (height, width, 3) BGR
def load_tif_bgr(path):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {path}")
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
    return img  # shape (height, width, 3), BGR

# Example: load a .tif file into sample
sample = load_tif_bgr("/Users/maryamasgarova/Desktop/graduation/matching algo/data_check/same_1/101_6.tif")
check_against = load_tif_bgr("/Users/maryamasgarova/Desktop/graduation/matching algo/data_check/same_1/101_7.tif")

In [3]:
def preprocess_image(image):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(16, 16))
    return clahe.apply(gray)
sample_preprocessed = preprocess_image(sample)
check_against_preprocessed = preprocess_image(check_against)

In [4]:
sift = cv2.SIFT_create()
keypoints_1, descriptors_1 = sift.detectAndCompute(sample_preprocessed, None)
keypoints_2, descriptors_2 = sift.detectAndCompute(check_against_preprocessed, None)
kp_data_1 = [(kp.pt[0], kp.pt[1], kp.size, kp.angle) for kp in keypoints_1]
print(len(kp_data_1), "keypoints,")
print("All keypoints (x, y, size, angle):")
print(kp_data_1)
desc_data = descriptors_1.tolist()
print("All descriptors:")
print(desc_data)


1367 keypoints,
All keypoints (x, y, size, angle):
[(2.6826658248901367, 94.70076751708984, 2.2095179557800293, 9.167518615722656), (4.180410861968994, 275.8724670410156, 2.8587002754211426, 144.62135314941406), (4.439975261688232, 271.91461181640625, 2.077444314956665, 197.5927734375), (4.677314758300781, 49.87871170043945, 2.1709702014923096, 27.846717834472656), (4.677314758300781, 49.87871170043945, 2.1709702014923096, 172.1487274169922), (5.012813568115234, 130.61643981933594, 4.054533958435059, 207.77626037597656), (5.049938678741455, 72.57756805419922, 2.14827823638916, 5.717620849609375), (5.674972057342529, 266.6695251464844, 5.212640285491943, 214.5443115234375), (5.848573684692383, 147.03448486328125, 4.918985366821289, 202.62266540527344), (6.176356792449951, 144.4727325439453, 6.522355556488037, 198.13409423828125), (6.6749467849731445, 208.7641143798828, 2.8938777446746826, 190.65756225585938), (7.064367771148682, 154.73109436035156, 3.2873451709747314, 210.85508728027344

In [5]:
flann = cv2.FlannBasedMatcher({'algorithm': 1, 'trees': 10}, {})
matches = flann.knnMatch(descriptors_1, descriptors_2, k=2)

In [6]:
match_points = [p for p, q in matches if p.distance < 0.7 * q.distance]

In [7]:
  # Filter matches using homography
if len(match_points) > 4:
        src_pts = np.float32([keypoints_1[m.queryIdx].pt for m in match_points]).reshape(-1, 1, 2)
        dst_pts = np.float32([keypoints_2[m.trainIdx].pt for m in match_points]).reshape(-1, 1, 2)
        H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 5.0)
        if mask is not None:
            match_points = [m for m, v in zip(match_points, mask.ravel()) if v == 1]


In [8]:
keypoints = min(len(keypoints_1), len(keypoints_2))

In [9]:
match_score = len(match_points) / keypoints * 100

In [10]:
kp1, kp2, mp = keypoints_1, keypoints_2, match_points
print("SCORE: " + str(match_score))


SCORE: 8.412582297000732


In [ ]:
image = check_against_preprocessed
result = cv2.drawMatches(sample, kp1, image, kp2, mp, None, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
result = cv2.resize(result, None, fx=4, fy=4)
cv2.imwrite("result.jpg", result)
cv2.imshow("RESULT", result)
cv2.waitKey(0)
cv2.destroyAllWindows()